In [0]:
from pyspark.sql.functions import col, trim, upper, regexp_replace, initcap, when, try_to_date, coalesce, year, row_number, desc,lower, round, lit, max
from pyspark.sql.window import Window

# Definindo schema
catalog_name = "cinedata_analytics"
schema_silver = "silver"
schema_bronze = "bronze"

# Criando schema se não existir
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_silver}")

# Lendo dados
df_origem_info_filmes = spark.read.table(f"{catalog_name}.{schema_bronze}.tb_movies_info")

# Renomeando colunas
df_silver_info_filmes = (
    df_origem_info_filmes
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

In [0]:
# Normalizando: maiúsculo, removendo hífens sobressalentes, espaços, e formato título
df_silver_info_filmes = df_silver_info_filmes.withColumn(
    "status_filme", initcap(trim(regexp_replace(upper(col("status_filme")), "-+", " ")))
)

# Normalizando: traduzindo status do filme para português e em caso de status desconhecido, informando não informado
df_silver_info_filmes = df_silver_info_filmes.withColumn(
    "status_filme",
    when(col("status_filme") == "Released", "Lançado")
    .when(col("status_filme") == "Post Production", "Pós-Produção")
    .when(col("status_filme") == "In Production", "Em Produção")
    .when(col("status_filme") == "Planned", "Planejado")
    .when(col("status_filme") == "Rumored", "Rumores")
    .when(col("status_filme") == "Canceled", "Cancelado")
    .otherwise("Não Informado")
)

In [0]:
# Convertendo campo de data lancamento testanto os diferentes padroes e caso não consiga converter, informando como campo null
df_silver_info_filmes = df_silver_info_filmes.withColumn(
    "data_lancamento",
    coalesce(
        try_to_date(col("data_lancamento"), "yyyy-MM-dd"),
        try_to_date(col("data_lancamento"), "MM-dd-yyyy"),
        try_to_date(col("data_lancamento"), "dd/MM/yyyy")
    )
)

# Extraindo somente o ano e criando uma coluna com o ano
df_silver_info_filmes = df_silver_info_filmes.withColumn(
    "ano_lancamento", year(col("data_lancamento"))
)

In [0]:
# Definindo
janela = Window.partitionBy("id_filme").orderBy(col("ingestion_datetime").desc())

# Mantendo somente a última versão da tabela e descartando colunas temporárias/auditoria
df_silver_info_filmes = (
    df_silver_info_filmes.withColumn(
        "linha_numero", row_number().over(janela)
    )
    .filter(col("linha_numero") == 1)
    .drop("linha_numero", "ingestion_datetime")
)
    

In [0]:
(
    df_silver_info_filmes.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog_name}.{schema_silver}.tb_info_filmes")
)

print("Tabela tb_info_filmes recriada com sucesso.")

In [0]:
#=====================================================================
#                       2° Tabela
#=====================================================================


In [0]:
df_cotacoes = spark.read.table(
    f"{catalog_name}.{schema_bronze}.tb_cotacao_dolar"
)

# 
cotacao_mais_recente = float(
    df_cotacoes.orderBy(col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)

print(f"Cotação carregada com sucesso: {cotacao_mais_recente}")

In [0]:
# Leitura da tabela de filmes
df_origem_financeiro = spark.read.table(
    f"{catalog_name}.{schema_bronze}.tb_movies_financials"
)

# Tratamento de strings e higienização monetária
termos_ausentes = ["unknown", "não informado", "nao informado", "n/a", "-", ""]

def limpar_campo_monetario(col_name):
    texto_limpo = lower(trim(col(col_name)))
    val_valido = when(
        texto_limpo.isin(termos_ausentes) | col(col_name).isNull(), lit(None)
    ).otherwise(col(col_name))

    apenas_valor = regexp_replace(val_valido, r"(?i)usd|\$|,|\s", "")

    multiplicador = (
        when(apenas_valor.rlike("(?i)B"), lit(1_000_000_000))
        .when(apenas_valor.rlike("(?i)M"), lit(1_000_000))
        .when(apenas_valor.rlike("(?i)K"), lit(1_000))
        .otherwise(lit(1))
    )

    num_puro = regexp_replace(apenas_valor, r"(?i)[a-z]", "").try_cast(
        "decimal(18,4)"
    )
    valor_final = (num_puro * multiplicador).cast("decimal(18,2)")

    return when(valor_final <= 0, lit(None)).otherwise(valor_final)


# JUSTIFICATIVA:
# Utilizamos groupBy + max() para consolidar os registros duplicados, pois os
# dados podem estar distribuídos entre diferentes linhas. Assim, conseguimos
# preservar os valores preenchidos em vez de escolher apenas o registro mais
# recente pelo ingestion_datetime.
df_tratado = (
    df_origem_financeiro.withColumnRenamed("id", "id_filme")
    .withColumn("orcamento_usd", limpar_campo_monetario("budget"))
    .withColumn("receita_usd", limpar_campo_monetario("revenue"))
    .groupBy("id_filme")
    .agg(
        max("orcamento_usd").alias("orcamento_usd"),
        max("receita_usd").alias("receita_usd"),
    )
)

In [0]:
# Aplicação do câmbio e cálculo das métricas de lucro
cotacao_usd = lit(cotacao_mais_recente)

# Cálculo das métricas financeiras em Real 
df_silver_financeiro = (
    df_tratado.withColumn(
        "orcamento_brl",
        (col("orcamento_usd") * cotacao_usd).cast("decimal(18,2)"),
    )
    .withColumn(
        "receita_brl", (col("receita_usd") * cotacao_usd).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_usd",
        (col("receita_usd") - col("orcamento_usd")).cast("decimal(18,2)"),
    )
    .withColumn(
        "lucro_brl",
        (col("receita_brl") - col("orcamento_brl")).cast("decimal(18,2)"),
    )
    .withColumn(
        "margem_lucro_percentual",
        when(
            (col("receita_usd").isNotNull())
            & (col("receita_usd") > 0)
            & (col("lucro_usd").isNotNull()),
            round((col("lucro_usd") / col("receita_usd")) * 100, 2),
        ).otherwise(lit(None)),
    )
)

# Gravação na camada Silver
(
    df_silver_financeiro.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog_name}.{schema_silver}.tb_financeiro_filmes")
)

print(
    f"Tabela tb_financeiro_filmes criada com sucesso com {df_silver_financeiro.count()} registos."
)


In [0]:
#=====================================================================
#                       3° Tabela
#=====================================================================

In [0]:
df_origem_metricas = spark.read.table(f"{catalog_name}.{schema_bronze}.tb_movies_metrics")

# Tratamento de strings e higienização monetária
df_silver_metricas = (
    df_origem_metricas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

# Converter coluna popularidade para double
df_silver_metricas = (
    df_silver_metricas
    .withColumn("popularidade", regexp_replace(col("popularidade"), ",", "."))
)

# Converter coluna popularidade para double
df_silver_metricas = (
    df_silver_metricas
    .withColumn("popularidade", col("popularidade").try_cast("double"))
)

# Verificar se a popularidade é maior ou igual a zero
df_silver_metricas = df_silver_metricas.withColumn(
    "popularidade",
    when(col("popularidade") >= 0, col("popularidade")).otherwise(None)
)

In [0]:
# Converter coluna nota_media_tmdb para double
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_tmdb",
    col("nota_media_tmdb").try_cast("double")
)

# Verificar se a nota_media_tmdb está entre 0 e 10
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_tmdb",
    when(col("nota_media_tmdb").between(0, 10), col("nota_media_tmdb")).otherwise(
        lit(None)
    ),
)

# Converter coluna nota_media_imdb para double
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_imdb",
    col("nota_media_imdb").try_cast("double")
)

# Verificar se a nota_media_imdb está entre 0 e 10
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_imdb",
    when(col("nota_media_imdb").between(0, 10), col("nota_media_imdb")).otherwise(
        lit(None)
    ),
)

In [0]:
# Converter colunas qtd_votos_tmdb e qtd_votos_imdb para
df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_tmdb",
    col("qtd_votos_tmdb").try_cast("integer")
)

# Verificar se qtd_votos_tmdb e qtd_votos_imdb são maiores ou iguais a zero
df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_imdb",
    col("qtd_votos_imdb").try_cast("integer")
)

# Verificar se qtd_votos_tmdb e qtd_votos_imdb são maiores ou iguais a zero
df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_imdb",
    when(col("qtd_votos_imdb") >= 0, col("qtd_votos_imdb")).otherwise(None)
)

df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_tmdb",
    when(col("qtd_votos_tmdb") >= 0, col("qtd_votos_tmdb")).otherwise(None)
)

# JUSTIFICATIVA:
# Usamos groupBy + max() para consolidar os registros duplicados, pois as
# informações podem estar preenchidas em linhas diferentes. Dessa forma,
# conseguimos reunir as métricas válidas em um único registro, evitando a perda
# de dados que poderia ocorrer ao considerar apenas o ingestion_datetime.
df_silver_metricas_final = df_silver_metricas.groupBy("id_filme").agg(
    max("popularidade").alias("popularidade"),
    max("nota_media_tmdb").alias("nota_media_tmdb"),
    max("qtd_votos_tmdb").alias("qtd_votos_tmdb"),
    max("nota_media_imdb").alias("nota_media_imdb"),
    max("qtd_votos_imdb").alias("qtd_votos_imdb"),
)

# Salvar dados
(
    df_silver_metricas_final.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog_name}.{schema_silver}.tb_metricas_engajamento")
)

In [0]:
#=====================================================================
#                       4° Tabela
#=====================================================================

In [0]:
df_origem_reviews = spark.read.table(f"{catalog_name}.{schema_bronze}.tb_movies_reviews")

df_silver_reviews = (
    df_origem_reviews
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)